# Digital coupon strip

Five annual digital coupons, $c\,\mathbb 1\{S_{t_i} \ge B\}$ paid at $t_i$, priced as five `DigitalOption`s
(Black–Scholes and `VanillaMCPricer`) and as the coupon leg of an `AutocallableOption` with the autocall disabled
(barrier at $10^6\%$) and no put (put barrier 0). Docs: *Models and Pricing* (Products: Digital options, Autocallable).

In [ ]:
import os
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorquant as tq

## Market

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
TICKER = "TICKER1"
eq = mkt.get_equity(TICKER)
K0 = round(float(eq.spot), 2)                                 # initial fixing
calendar, mf = tq.TARGET(), tq.BusinessDayConvention.ModifiedFollowing
print(f"{EVALUATION_DATE}  {TICKER} spot {float(eq.spot):.2f}")

## Strip
Coupon 5% of the notional, barrier $B$ = 90% of $K_0$, annual fixings = payment dates.

In [ ]:
COUPON, BARRIER_PCT = 5.0, 90.0
B = BARRIER_PCT / 100 * K0
dates = [calendar.advance(EVALUATION_DATE, i, tq.TimeUnit.Years, mf) for i in range(1, 6)]


def digital(expiry):
    return tq.DigitalOption(eq.currency, EVALUATION_DATE, expiry, tq.OptionType.Call, B,
                            payoff_type=tq.PayoffType.CashOrNothing, cash_amount=COUPON / 100, underlying=TICKER)


def price(pricer, product):
    pricer.price(product, mkt)
    return float(product.price)

## Digitals one by one
- Black–Scholes: $c\,P(0,t_i)\,N(d_2)$ with the vol at $(B, t_i)$;
- MC GBM (hard indicator): the same model, one exact step;
- MC local vol (hard indicator, weekly steps): the whole smile;
- call spread on the surface $c\,\big(C(B - h) - C(B + h)\big)/2h$, $h$ = 1% of $B$: the smile slope included, model-free.

In [ ]:
def call(expiry, strike):
    return tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=expiry, option_type=tq.OptionType.Call,
                            strike=strike, underlying=TICKER)


h = 0.01 * B
rows = []
for d in dates:
    T = (d - EVALUATION_DATE).days / 365
    bs = price(tq.BlackScholesPricer(), digital(d))
    gbm = price(tq.VanillaMCPricer(n_paths=200_000, n_steps=1, digital_smoothing=0.0), digital(d))
    lv = price(tq.VanillaMCPricer(model=tq.LocalVolatilityModel, n_paths=100_000, n_steps=max(int(52 * T), 1),
                                  digital_smoothing=0.0), digital(d))
    spread = COUPON / 100 * (price(tq.BlackScholesPricer(), call(d, B - h))
                             - price(tq.BlackScholesPricer(), call(d, B + h))) / (2 * h)
    rows.append((d, bs, gbm, lv, spread))
digitals = pd.DataFrame(rows, columns=["date", "BS", "MC GBM", "MC LV", "call spread"]).set_index("date")
print((100 * digitals).round(3).to_string())
print("\nstrip (% notional):")
print((100 * digitals.sum()).round(3).to_string())

## The same strip as an autocallable
`AutocallableMCPricer` (local vol, hard barriers): `coupon_pv` is the coupon leg; per coupon from the same paths
(`pricer.paths` at `pricer.date_grid`): $c\,P(0,t_i)\,\overline{\mathbb 1\{S_{t_i} \ge B\}}$.

In [ ]:
NEVER = 1e6      # autocall barrier in % of K0: never reached


def autocallable_template(fixing_dates, coupon, coupon_barrier, put_barrier):
    # AutocallableOption per unit notional; autocall disabled, the put active only if put_barrier > 0
    n = len(fixing_dates)
    return tq.AutocallableOption(
        ccy=eq.currency, notional=1.0, start_date=EVALUATION_DATE, end_date=fixing_dates[-1], strike=K0,
        coupon_fixing_dates=fixing_dates, coupon_payment_dates=fixing_dates,
        coupon_rates=[coupon] * n, coupon_barriers=[coupon_barrier] * n, memory=False,
        autocall_fixing_dates=fixing_dates, autocall_payment_dates=fixing_dates, autocall_barrier=[NEVER] * n,
        payoff_barrier=put_barrier, payoff_participation=1.0, payoff_type="put", underlying=TICKER)

In [ ]:
def strip_as_autocallable(model):
    product = autocallable_template(dates, COUPON, BARRIER_PCT, put_barrier=0.0)
    pricer = tq.AutocallableMCPricer(model=model, n_paths=100_000, barrier_smoothing=0.0)
    pricer.price(product, mkt)
    paths, col = pricer.paths.numpy(), {d: j for j, d in enumerate(pricer.date_grid)}
    per_coupon = [COUPON / 100 * float(eq.discount_curve.discount(d)) * np.mean(paths[:, col[d]] >= B) for d in dates]
    return float(product.price), float(product.put_pv), per_coupon, float(product.standard_error)


ac_lv, put_lv, coupons_lv, se_lv = strip_as_autocallable(tq.LocalVolatilityModel)
digitals["AC LV"] = coupons_lv
print((100 * digitals).round(3).to_string())
print(f"\nautocallable (LV) NPV  : {100 * ac_lv:.3f}%  +/- {100 * se_lv:.3f}%   (put leg {100 * put_lv:.1e})")
print(f"digital strip, MC LV   : {100 * digitals['MC LV'].sum():.3f}%")
print(f"digital strip, BS      : {100 * digitals['BS'].sum():.3f}%")

## Reading
- MC GBM = BS: the same model.
- AC LV = MC LV within the Monte Carlo error: with the autocall off and no put the autocallable is the digital strip.
- LV and the call spread above BS: a negative skew raises the digital call by $-\mathcal V\,\partial\sigma/\partial K$, which the
  BS digital at the vol of the strike misses.
- Not with GBM: `GeometricBrownianMotion.from_equity` is the Black–Scholes model of one expiry (vol at $(K, T)$, spot
  escrowed of all the dividends up to $T$), so on a multi-date product it is wrong at the intermediate dates.